In [1]:
import gc
import itertools
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import f_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [2]:
DATASET_NAME = "CLAP 0.5s"

DATASET_PATH = Path(
    "/Users/bhavaykhatri/Desktop/msclap_2023/"
    "singBAP_dataset_clap-2023_0.5s.parquet"
)

OUTPUT_DIR = Path(
    "/Users/bhavaykhatri/Desktop/msclap_2023/"
    "clap_0.5s_all_feature_selection"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET_CLASSES = [
    "correct",
    "arched_back",
    "hunched_back",
    "sideways",
    "chest_breathing",
    "over_articulation",
    "under_articulation",
]

TARGET_EXPERIENCE = [
    "intermediate",
    "professional",
]

RANDOM_STATE = 42
INNER_RANDOM_STATE = 43

ORIGINAL_FEATURE_COUNT = 1024

# Fixed feature counts for fair ANOVA vs L1 comparison
ANOVA_K_VALUES = [
    512,
    768,
    896,
    960,
    992,
]

L1_K_VALUES = [
    512,
    768,
    896,
    960,
    992,
]

# One L1 model ranks all features
L1_RANKING_C = 0.03

# Sequential selection searches only these top ANOVA candidates
SFS_PREFILTER_K = 30
SFS_MAXIMUM_SELECTED = 15

# Brute force searches all size-3 and size-4 subsets
# among the top eight ANOVA candidates
BRUTE_FORCE_TOP_K = 8
BRUTE_FORCE_MINIMUM_SIZE = 3
BRUTE_FORCE_MAXIMUM_SIZE = 4

# Fast proxy model used during feature search
SEARCH_C = 0.1
SEARCH_TOL = 1e-2
SEARCH_MAX_ITER = 3000

BASELINE_ACCURACY = 0.3762
BASELINE_BALANCED_ACCURACY = 0.3828
BASELINE_MACRO_F1 = 0.3790

SELECTION_CHECKPOINT_PATH = (
    OUTPUT_DIR
    / "selection_checkpoint.csv"
)

MLP_CHECKPOINT_PATH = (
    OUTPUT_DIR
    / "mlp_validation_checkpoint.csv"
)

print("Dataset:", DATASET_NAME)
print("Dataset path:", DATASET_PATH)
print("Output directory:", OUTPUT_DIR.resolve())

Dataset: CLAP 0.5s
Dataset path: /Users/bhavaykhatri/Desktop/msclap_2023/singBAP_dataset_clap-2023_0.5s.parquet
Output directory: /Users/bhavaykhatri/Desktop/msclap_2023/clap_0.5s_all_feature_selection


In [3]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )

df = pd.read_parquet(DATASET_PATH)

required_columns = {
    "embedding",
    "condition",
    "experience",
    "filename",
}

missing_columns = (
    required_columns
    - set(df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing columns: {sorted(missing_columns)}"
    )

print("Original shape:", df.shape)

df = df[
    df["experience"].isin(
        TARGET_EXPERIENCE
    )
].copy()

df = df[
    df["condition"].isin(
        TARGET_CLASSES
    )
].copy()

df = df.reset_index(drop=True)

print("Filtered shape:", df.shape)

display(
    df["condition"]
    .value_counts()
    .reindex(TARGET_CLASSES)
    .rename_axis("Class")
    .to_frame("Samples")
)

Original shape: (34409, 13)
Filtered shape: (28418, 13)


,Samples
Class,
correct,5137
arched_back,3362
hunched_back,4540
sideways,4285
chest_breathing,4259
over_articulation,3454
under_articulation,3381


In [4]:
def decode_embedding(value):
    if isinstance(
        value,
        (bytes, bytearray, memoryview),
    ):
        return np.frombuffer(
            value,
            dtype=np.float32,
        ).copy()

    return np.asarray(
        value,
        dtype=np.float32,
    ).reshape(-1)


X = np.ascontiguousarray(
    np.vstack(
        df["embedding"].map(
            decode_embedding
        )
    ),
    dtype=np.float32,
)

y = (
    df["condition"]
    .astype(str)
    .to_numpy()
)

groups = (
    df["filename"]
    .astype(str)
    .to_numpy()
)

valid_rows = np.isfinite(X).all(
    axis=1
)

invalid_rows = int(
    len(X) - valid_rows.sum()
)

print("Invalid rows:", invalid_rows)

if invalid_rows > 0:
    X = X[valid_rows]
    y = y[valid_rows]
    groups = groups[valid_rows]

print("Features:", X.shape)
print("Labels:", y.shape)
print("Unique recordings:", len(np.unique(groups)))

if X.shape[1] != ORIGINAL_FEATURE_COUNT:
    raise ValueError(
        f"Expected {ORIGINAL_FEATURE_COUNT} features, "
        f"but found {X.shape[1]}."
    )

del df
gc.collect()

Invalid rows: 0
Features: (28418, 1024)
Labels: (28418,)
Unique recordings: 3046


0

In [5]:
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

outer_train_idx, test_idx = next(
    outer_splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_outer_train = X[
    outer_train_idx
]

y_outer_train = y[
    outer_train_idx
]

outer_train_groups = groups[
    outer_train_idx
]

X_test = X[test_idx]
y_test = y[test_idx]
test_groups = groups[test_idx]

outer_overlap = (
    set(outer_train_groups)
    & set(test_groups)
)

inner_splitter = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=INNER_RANDOM_STATE,
)

feature_train_relative_idx, validation_relative_idx = next(
    inner_splitter.split(
        X_outer_train,
        y_outer_train,
        groups=outer_train_groups,
    )
)

X_feature_train = X_outer_train[
    feature_train_relative_idx
]

y_feature_train = y_outer_train[
    feature_train_relative_idx
]

feature_train_groups = outer_train_groups[
    feature_train_relative_idx
]

X_validation = X_outer_train[
    validation_relative_idx
]

y_validation = y_outer_train[
    validation_relative_idx
]

validation_groups = outer_train_groups[
    validation_relative_idx
]

inner_overlap = (
    set(feature_train_groups)
    & set(validation_groups)
)

print("Full dataset:", X.shape)
print("Outer training:", X_outer_train.shape)
print("Final test:", X_test.shape)
print()
print("Feature-search training:", X_feature_train.shape)
print("Validation:", X_validation.shape)
print()
print("Shared outer recordings:", len(outer_overlap))
print("Shared inner recordings:", len(inner_overlap))

assert len(outer_overlap) == 0
assert len(inner_overlap) == 0

del X, y, groups
gc.collect()

Full dataset: (28418, 1024)
Outer training: (22735, 1024)
Final test: (5683, 1024)

Feature-search training: (17049, 1024)
Validation: (5686, 1024)

Shared outer recordings: 0
Shared inner recordings: 0


0

In [6]:
search_scaler = StandardScaler()

X_feature_train_scaled = (
    search_scaler
    .fit_transform(X_feature_train)
    .astype(
        np.float32,
        copy=False,
    )
)

X_validation_scaled = (
    search_scaler
    .transform(X_validation)
    .astype(
        np.float32,
        copy=False,
    )
)

score_cache = {}


def evaluate_feature_indices(
    feature_indices,
):
    feature_indices = np.unique(
        np.asarray(
            feature_indices,
            dtype=np.int32,
        )
    )

    if feature_indices.size == 0:
        return np.nan

    cache_key = tuple(
        feature_indices.tolist()
    )

    if cache_key in score_cache:
        return score_cache[cache_key]

    model = LinearSVC(
        C=SEARCH_C,
        penalty="l2",
        loss="squared_hinge",
        class_weight="balanced",
        dual=False,
        tol=SEARCH_TOL,
        max_iter=SEARCH_MAX_ITER,
        random_state=RANDOM_STATE,
    )

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            ConvergenceWarning,
        )

        model.fit(
            X_feature_train_scaled[
                :,
                feature_indices,
            ],
            y_feature_train,
        )

    predictions = model.predict(
        X_validation_scaled[
            :,
            feature_indices,
        ]
    )

    macro_f1 = f1_score(
        y_validation,
        predictions,
        average="macro",
        zero_division=0,
    )

    score_cache[cache_key] = macro_f1

    return macro_f1

In [7]:
selection_results = []
selected_feature_sets = {}


def record_selection(
    method,
    family,
    feature_indices,
    validation_macro_f1,
    elapsed_time,
):
    feature_indices = np.unique(
        np.asarray(
            feature_indices,
            dtype=np.int32,
        )
    )

    selected_count = len(
        feature_indices
    )

    result = {
        "Method": method,
        "Family": family,
        "Selected Features": selected_count,
        "Features Removed": (
            ORIGINAL_FEATURE_COUNT
            - selected_count
        ),
        "Feature Reduction (%)": (
            1
            - (
                selected_count
                / ORIGINAL_FEATURE_COUNT
            )
        ) * 100,
        "Proxy Validation Macro F1": (
            validation_macro_f1
        ),
        "Selection Time (s)": (
            elapsed_time
        ),
    }

    selection_results.append(result)

    selected_feature_sets[
        method
    ] = feature_indices.copy()

    pd.DataFrame(
        selection_results
    ).to_csv(
        SELECTION_CHECKPOINT_PATH,
        index=False,
    )

    print(
        f"{method}: "
        f"{selected_count} features, "
        f"Macro F1={validation_macro_f1:.4f}, "
        f"time={elapsed_time:.2f}s"
    )

In [8]:
all_feature_indices = np.arange(
    ORIGINAL_FEATURE_COUNT,
    dtype=np.int32,
)

start_time = time.time()

all_features_proxy_f1 = (
    evaluate_feature_indices(
        all_feature_indices
    )
)

record_selection(
    method="All Features",
    family="Baseline",
    feature_indices=all_feature_indices,
    validation_macro_f1=all_features_proxy_f1,
    elapsed_time=time.time() - start_time,
)


anova_start = time.time()

with warnings.catch_warnings():
    warnings.simplefilter(
        "ignore",
        RuntimeWarning,
    )

    anova_scores, _ = f_classif(
        X_feature_train,
        y_feature_train,
    )

anova_scores = np.nan_to_num(
    anova_scores,
    nan=-np.inf,
    posinf=np.finfo(np.float32).max,
    neginf=-np.inf,
)

anova_ranked_indices = np.argsort(
    anova_scores
)[::-1].astype(np.int32)

print(
    "ANOVA ranking time:",
    f"{time.time() - anova_start:.2f}s",
)

for k in ANOVA_K_VALUES:
    start_time = time.time()

    selected_indices = (
        anova_ranked_indices[:k]
    )

    validation_f1 = (
        evaluate_feature_indices(
            selected_indices
        )
    )

    record_selection(
        method=f"ANOVA K={k}",
        family="ANOVA",
        feature_indices=selected_indices,
        validation_macro_f1=validation_f1,
        elapsed_time=time.time() - start_time,
    )

All Features: 1024 features, Macro F1=0.3063, time=131.88s
ANOVA ranking time: 0.10s
ANOVA K=512: 512 features, Macro F1=0.2737, time=30.59s
ANOVA K=768: 768 features, Macro F1=0.2977, time=66.94s
ANOVA K=896: 896 features, Macro F1=0.2975, time=87.63s
ANOVA K=960: 960 features, Macro F1=0.3027, time=109.97s
ANOVA K=992: 992 features, Macro F1=0.3021, time=109.89s


In [9]:
l1_start = time.time()

l1_ranker = LinearSVC(
    C=L1_RANKING_C,
    penalty="l1",
    loss="squared_hinge",
    class_weight="balanced",
    dual=False,
    tol=1e-2,
    max_iter=5000,
    random_state=RANDOM_STATE,
)

with warnings.catch_warnings():
    warnings.simplefilter(
        "ignore",
        ConvergenceWarning,
    )

    l1_ranker.fit(
        X_feature_train_scaled,
        y_feature_train,
    )

l1_feature_importance = np.linalg.norm(
    l1_ranker.coef_,
    ord=2,
    axis=0,
)

l1_ranked_indices = np.argsort(
    l1_feature_importance
)[::-1].astype(np.int32)

print(
    "L1 ranking time:",
    f"{time.time() - l1_start:.2f}s",
)

print(
    "L1 iterations:",
    l1_ranker.n_iter_,
)

for k in L1_K_VALUES:
    start_time = time.time()

    selected_indices = (
        l1_ranked_indices[:k]
    )

    validation_f1 = (
        evaluate_feature_indices(
            selected_indices
        )
    )

    record_selection(
        method=f"L1 Ranked K={k}",
        family="L1 Ranked",
        feature_indices=selected_indices,
        validation_macro_f1=validation_f1,
        elapsed_time=time.time() - start_time,
    )

L1 ranking time: 29.46s
L1 iterations: 574
L1 Ranked K=512: 512 features, Macro F1=0.2787, time=20.51s
L1 Ranked K=768: 768 features, Macro F1=0.2937, time=53.01s
L1 Ranked K=896: 896 features, Macro F1=0.3012, time=94.71s
L1 Ranked K=960: 960 features, Macro F1=0.2999, time=99.77s
L1 Ranked K=992: 992 features, Macro F1=0.3042, time=103.45s


In [10]:
def sequential_forward_selection(
    candidate_indices,
    maximum_selected=15,
):
    candidate_indices = list(
        map(int, candidate_indices)
    )

    selected = []
    remaining = candidate_indices.copy()

    best_overall_score = -np.inf
    best_overall_subset = None

    history = []

    maximum_steps = min(
        maximum_selected,
        len(candidate_indices),
    )

    for step in range(
        maximum_steps
    ):
        best_step_feature = None
        best_step_score = -np.inf

        for feature_index in remaining:
            trial_subset = (
                selected
                + [feature_index]
            )

            score = (
                evaluate_feature_indices(
                    trial_subset
                )
            )

            if score > best_step_score:
                best_step_score = score
                best_step_feature = (
                    feature_index
                )

        selected.append(
            best_step_feature
        )

        remaining.remove(
            best_step_feature
        )

        history.append({
            "Step": step + 1,
            "Added Feature": (
                best_step_feature
            ),
            "Selected Features": (
                len(selected)
            ),
            "Validation Macro F1": (
                best_step_score
            ),
        })

        print(
            f"Step {step + 1}: "
            f"added {best_step_feature}, "
            f"Macro F1={best_step_score:.4f}"
        )

        if (
            best_step_score
            > best_overall_score
        ):
            best_overall_score = (
                best_step_score
            )

            best_overall_subset = (
                selected.copy()
            )

    return (
        np.asarray(
            best_overall_subset,
            dtype=np.int32,
        ),
        best_overall_score,
        pd.DataFrame(history),
    )

In [11]:
sfs_candidate_indices = (
    anova_ranked_indices[
        :SFS_PREFILTER_K
    ]
)

print(
    "SFS candidate features:",
    len(sfs_candidate_indices),
)

sfs_start = time.time()

(
    sequential_indices,
    sequential_validation_f1,
    sequential_history_df,
) = sequential_forward_selection(
    candidate_indices=(
        sfs_candidate_indices
    ),
    maximum_selected=(
        SFS_MAXIMUM_SELECTED
    ),
)

record_selection(
    method="Sequential Forward",
    family="Sequential",
    feature_indices=sequential_indices,
    validation_macro_f1=(
        sequential_validation_f1
    ),
    elapsed_time=(
        time.time() - sfs_start
    ),
)

display(sequential_history_df)

sequential_history_df.to_csv(
    OUTPUT_DIR
    / "sequential_forward_history.csv",
    index=False,
)

SFS candidate features: 30
Step 1: added 330, Macro F1=0.1123
Step 2: added 727, Macro F1=0.1462
Step 3: added 496, Macro F1=0.1642
Step 4: added 1013, Macro F1=0.1717
Step 5: added 179, Macro F1=0.1751
Step 6: added 864, Macro F1=0.1768
Step 7: added 131, Macro F1=0.1817
Step 8: added 588, Macro F1=0.1823
Step 9: added 738, Macro F1=0.1830
Step 10: added 201, Macro F1=0.1838
Step 11: added 422, Macro F1=0.1829
Step 12: added 640, Macro F1=0.1827
Step 13: added 297, Macro F1=0.1812
Step 14: added 849, Macro F1=0.1857
Step 15: added 924, Macro F1=0.1851
Sequential Forward: 14 features, Macro F1=0.1857, time=23.54s


,Step,Added Feature,Selected Features,Validation Macro F1
0,1,330,1,0.112311
1,2,727,2,0.146230
2,3,496,3,0.164180
3,4,1013,4,0.171688
4,5,179,5,0.175068
5,6,864,6,0.176809
6,7,131,7,0.181731
7,8,588,8,0.182288
8,9,738,9,0.183026
9,10,201,10,0.183769


In [12]:
def exhaustive_feature_selection(
    candidate_indices,
    minimum_subset_size=3,
    maximum_subset_size=4,
):
    candidate_indices = list(
        map(int, candidate_indices)
    )

    best_score = -np.inf
    best_subset = None
    evaluated_subsets = 0

    history = []

    for subset_size in range(
        minimum_subset_size,
        maximum_subset_size + 1,
    ):
        print(
            f"Testing subsets of size "
            f"{subset_size}..."
        )

        for subset in itertools.combinations(
            candidate_indices,
            subset_size,
        ):
            score = (
                evaluate_feature_indices(
                    subset
                )
            )

            evaluated_subsets += 1

            if score > best_score:
                best_score = score
                best_subset = subset

                history.append({
                    "Evaluated Subsets": (
                        evaluated_subsets
                    ),
                    "Subset Size": (
                        subset_size
                    ),
                    "Validation Macro F1": (
                        score
                    ),
                    "Feature Indices": (
                        list(subset)
                    ),
                })

                print(
                    f"New best: "
                    f"F1={score:.4f}, "
                    f"features={subset}"
                )

    return (
        np.asarray(
            best_subset,
            dtype=np.int32,
        ),
        best_score,
        evaluated_subsets,
        pd.DataFrame(history),
    )

In [13]:
brute_candidate_indices = (
    anova_ranked_indices[
        :BRUTE_FORCE_TOP_K
    ]
)

print(
    "Brute-force candidates:",
    brute_candidate_indices,
)

brute_start = time.time()

(
    brute_force_indices,
    brute_force_validation_f1,
    evaluated_subsets,
    brute_force_history_df,
) = exhaustive_feature_selection(
    candidate_indices=(
        brute_candidate_indices
    ),
    minimum_subset_size=(
        BRUTE_FORCE_MINIMUM_SIZE
    ),
    maximum_subset_size=(
        BRUTE_FORCE_MAXIMUM_SIZE
    ),
)

record_selection(
    method="Brute Force",
    family="Exhaustive",
    feature_indices=brute_force_indices,
    validation_macro_f1=(
        brute_force_validation_f1
    ),
    elapsed_time=(
        time.time() - brute_start
    ),
)

print(
    "Subsets evaluated:",
    evaluated_subsets,
)

display(brute_force_history_df)

brute_force_history_df.to_csv(
    OUTPUT_DIR
    / "brute_force_history.csv",
    index=False,
)

Brute-force candidates: [727  95 864 389 554 179 785 893]
Testing subsets of size 3...
New best: F1=0.1467, features=(727, 95, 864)
New best: F1=0.1478, features=(727, 95, 554)
New best: F1=0.1587, features=(727, 864, 389)
New best: F1=0.1591, features=(864, 554, 179)
Testing subsets of size 4...
New best: F1=0.1602, features=(727, 864, 389, 554)
New best: F1=0.1611, features=(727, 389, 554, 179)
New best: F1=0.1636, features=(864, 389, 554, 179)
Brute Force: 4 features, Macro F1=0.1636, time=4.25s
Subsets evaluated: 126


,Evaluated Subsets,Subset Size,Validation Macro F1,Feature Indices
0,1,3,0.146666,"[727, 95, 864]"
1,3,3,0.147800,"[727, 95, 554]"
2,7,3,0.158682,"[727, 864, 389]"
3,41,3,0.159129,"[864, 554, 179]"
4,72,4,0.160155,"[727, 864, 389, 554]"
5,82,4,0.161097,"[727, 389, 554, 179]"
6,112,4,0.163563,"[864, 389, 554, 179]"


In [14]:
selection_results_df = pd.DataFrame(
    selection_results
)

selection_results_df = (
    selection_results_df
    .sort_values(
        "Proxy Validation Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(selection_results_df)

selection_results_df.to_csv(
    OUTPUT_DIR
    / "all_selection_results.csv",
    index=False,
)

,Method,Family,Selected Features,Features Removed,Feature Reduction (%),Proxy Validation Macro F1,Selection Time (s)
0,All Features,Baseline,1024,0,0.000000,0.306294,131.881166
1,L1 Ranked K=992,L1 Ranked,992,32,3.125000,0.304206,103.454557
2,ANOVA K=960,ANOVA,960,64,6.250000,0.302659,109.966467
3,ANOVA K=992,ANOVA,992,32,3.125000,0.302145,109.889681
4,L1 Ranked K=896,L1 Ranked,896,128,12.500000,0.301210,94.709854
5,L1 Ranked K=960,L1 Ranked,960,64,6.250000,0.299912,99.766526
6,ANOVA K=768,ANOVA,768,256,25.000000,0.297714,66.942958
7,ANOVA K=896,ANOVA,896,128,12.500000,0.297499,87.628598
8,L1 Ranked K=768,L1 Ranked,768,256,25.000000,0.293687,53.014936
9,L1 Ranked K=512,L1 Ranked,512,512,50.000000,0.278701,20.505344


In [15]:
best_family_indices = (
    selection_results_df
    .groupby("Family")[
        "Proxy Validation Macro F1"
    ]
    .idxmax()
)

best_family_rows = (
    selection_results_df
    .loc[best_family_indices]
    .sort_values(
        "Proxy Validation Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(best_family_rows)

FINAL_CANDIDATE_FEATURE_SETS = {
    row["Method"]: (
        selected_feature_sets[
            row["Method"]
        ]
    )
    for _, row
    in best_family_rows.iterrows()
}

print("Candidates moving to MLP validation:")

for method_name, indices in (
    FINAL_CANDIDATE_FEATURE_SETS.items()
):
    print(
        method_name,
        "->",
        len(indices),
        "features",
    )

,Method,Family,Selected Features,Features Removed,Feature Reduction (%),Proxy Validation Macro F1,Selection Time (s)
0,All Features,Baseline,1024,0,0.000000,0.306294,131.881166
1,L1 Ranked K=992,L1 Ranked,992,32,3.125000,0.304206,103.454557
2,ANOVA K=960,ANOVA,960,64,6.250000,0.302659,109.966467
3,Sequential Forward,Sequential,14,1010,98.632812,0.185712,23.542727
4,Brute Force,Exhaustive,4,1020,99.609375,0.163563,4.251683


Candidates moving to MLP validation:
All Features -> 1024 features
L1 Ranked K=992 -> 992 features
ANOVA K=960 -> 960 features
Sequential Forward -> 14 features
Brute Force -> 4 features


In [16]:
def build_baseline_mlp():
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(
                256,
                128,
            ),
            early_stopping=True,
            max_iter=300,
            random_state=RANDOM_STATE,
        ),
    )

In [17]:
mlp_validation_results = []

for method_name, feature_indices in (
    FINAL_CANDIDATE_FEATURE_SETS.items()
):
    print("\n" + "=" * 70)

    print(
        method_name,
        "->",
        len(feature_indices),
        "features",
    )

    model = build_baseline_mlp()

    start_time = time.time()

    with warnings.catch_warnings(
        record=True
    ) as caught_warnings:
        warnings.simplefilter(
            "always",
            ConvergenceWarning,
        )

        model.fit(
            X_feature_train[
                :,
                feature_indices,
            ],
            y_feature_train,
        )

    predictions = model.predict(
        X_validation[
            :,
            feature_indices,
        ]
    )

    elapsed_time = (
        time.time() - start_time
    )

    mlp_model = model.named_steps[
        "mlpclassifier"
    ]

    result = {
        "Method": method_name,
        "Selected Features": (
            len(feature_indices)
        ),
        "Features Removed": (
            ORIGINAL_FEATURE_COUNT
            - len(feature_indices)
        ),
        "Feature Reduction (%)": (
            1
            - (
                len(feature_indices)
                / ORIGINAL_FEATURE_COUNT
            )
        ) * 100,
        "Validation Accuracy": (
            accuracy_score(
                y_validation,
                predictions,
            )
        ),
        "Validation Balanced Accuracy": (
            balanced_accuracy_score(
                y_validation,
                predictions,
            )
        ),
        "Validation Macro F1": (
            f1_score(
                y_validation,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "MLP Iterations": int(
            mlp_model.n_iter_
        ),
        "Train/Eval Time (s)": (
            elapsed_time
        ),
    }

    mlp_validation_results.append(
        result
    )

    pd.DataFrame(
        mlp_validation_results
    ).to_csv(
        MLP_CHECKPOINT_PATH,
        index=False,
    )

    print(
        "Accuracy:",
        f"{result['Validation Accuracy']:.4f}",
    )

    print(
        "Balanced accuracy:",
        f"{result['Validation Balanced Accuracy']:.4f}",
    )

    print(
        "Macro F1:",
        f"{result['Validation Macro F1']:.4f}",
    )

    print(
        "Iterations:",
        result["MLP Iterations"],
    )

    print(
        "Time:",
        f"{elapsed_time:.2f}s",
    )

    del model, predictions
    gc.collect()


All Features -> 1024 features
Accuracy: 0.3500
Balanced accuracy: 0.3505
Macro F1: 0.3497
Iterations: 33
Time: 25.38s

L1 Ranked K=992 -> 992 features
Accuracy: 0.3375
Balanced accuracy: 0.3403
Macro F1: 0.3445
Iterations: 38
Time: 30.29s

ANOVA K=960 -> 960 features
Accuracy: 0.3380
Balanced accuracy: 0.3414
Macro F1: 0.3447
Iterations: 48
Time: 37.57s

Sequential Forward -> 14 features
Accuracy: 0.2160
Balanced accuracy: 0.2142
Macro F1: 0.2126
Iterations: 36
Time: 14.22s

Brute Force -> 4 features
Accuracy: 0.1877
Balanced accuracy: 0.1710
Macro F1: 0.1504
Iterations: 26
Time: 11.01s


In [18]:
mlp_validation_results_df = pd.DataFrame(
    mlp_validation_results
)

mlp_validation_results_df = (
    mlp_validation_results_df
    .sort_values(
        [
            "Validation Macro F1",
            "Validation Balanced Accuracy",
            "Validation Accuracy",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(mlp_validation_results_df)

BEST_CONFIGURATION = (
    mlp_validation_results_df
    .iloc[0]
    .to_dict()
)

BEST_METHOD = str(
    BEST_CONFIGURATION["Method"]
)

print("Winning method:", BEST_METHOD)

print(
    "Selected features:",
    BEST_CONFIGURATION[
        "Selected Features"
    ],
)

print(
    "Feature reduction:",
    f"{BEST_CONFIGURATION['Feature Reduction (%)']:.2f}%",
)

print(
    "Validation Macro F1:",
    f"{BEST_CONFIGURATION['Validation Macro F1']:.4f}",
)

,Method,Selected Features,Features Removed,Feature Reduction (%),Validation Accuracy,Validation Balanced Accuracy,Validation Macro F1,MLP Iterations,Train/Eval Time (s)
0,All Features,1024,0,0.000000,0.349982,0.350544,0.349706,33,25.382765
1,ANOVA K=960,960,64,6.250000,0.338023,0.341439,0.344656,48,37.573766
2,L1 Ranked K=992,992,32,3.125000,0.337496,0.340327,0.344520,38,30.294928
3,Sequential Forward,14,1010,98.632812,0.215969,0.214213,0.212601,36,14.218582
4,Brute Force,4,1020,99.609375,0.187654,0.171036,0.150357,26,11.007747


Winning method: All Features
Selected features: 1024
Feature reduction: 0.00%
Validation Macro F1: 0.3497


In [19]:
def calculate_anova_ranking(
    X_training,
    y_training,
):
    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            RuntimeWarning,
        )

        scores, _ = f_classif(
            X_training,
            y_training,
        )

    scores = np.nan_to_num(
        scores,
        nan=-np.inf,
        posinf=np.finfo(np.float32).max,
        neginf=-np.inf,
    )

    return np.argsort(
        scores
    )[::-1].astype(np.int32)


def calculate_l1_ranking(
    X_training,
    y_training,
):
    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X_training
    )

    model = LinearSVC(
        C=L1_RANKING_C,
        penalty="l1",
        loss="squared_hinge",
        class_weight="balanced",
        dual=False,
        tol=1e-2,
        max_iter=5000,
        random_state=RANDOM_STATE,
    )

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            ConvergenceWarning,
        )

        model.fit(
            X_scaled,
            y_training,
        )

    importance = np.linalg.norm(
        model.coef_,
        ord=2,
        axis=0,
    )

    return np.argsort(
        importance
    )[::-1].astype(np.int32)


if BEST_METHOD == "All Features":
    final_feature_indices = (
        np.arange(
            ORIGINAL_FEATURE_COUNT,
            dtype=np.int32,
        )
    )

elif BEST_METHOD.startswith(
    "ANOVA K="
):
    final_k = int(
        BEST_METHOD.rsplit(
            "=",
            maxsplit=1,
        )[1]
    )

    final_ranking = (
        calculate_anova_ranking(
            X_outer_train,
            y_outer_train,
        )
    )

    final_feature_indices = (
        final_ranking[:final_k]
    )

elif BEST_METHOD.startswith(
    "L1 Ranked K="
):
    final_k = int(
        BEST_METHOD.rsplit(
            "=",
            maxsplit=1,
        )[1]
    )

    final_ranking = (
        calculate_l1_ranking(
            X_outer_train,
            y_outer_train,
        )
    )

    final_feature_indices = (
        final_ranking[:final_k]
    )

else:
    # Sequential Forward or Brute Force:
    # keep the subset chosen through inner validation.
    final_feature_indices = (
        FINAL_CANDIDATE_FEATURE_SETS[
            BEST_METHOD
        ].copy()
    )

final_feature_indices = np.asarray(
    final_feature_indices,
    dtype=np.int32,
)

FEATURES_REMOVED = (
    ORIGINAL_FEATURE_COUNT
    - len(final_feature_indices)
)

FEATURE_REDUCTION_PERCENT = (
    FEATURES_REMOVED
    / ORIGINAL_FEATURE_COUNT
) * 100

print("Final method:", BEST_METHOD)
print("Features kept:", len(final_feature_indices))
print("Features removed:", FEATURES_REMOVED)

print(
    "Feature reduction:",
    f"{FEATURE_REDUCTION_PERCENT:.2f}%",
)

Final method: All Features
Features kept: 1024
Features removed: 0
Feature reduction: 0.00%


In [20]:
final_model = build_baseline_mlp()

final_start = time.time()

with warnings.catch_warnings(
    record=True
) as final_warnings:
    warnings.simplefilter(
        "always",
        ConvergenceWarning,
    )

    final_model.fit(
        X_outer_train[
            :,
            final_feature_indices,
        ],
        y_outer_train,
    )

test_predictions = final_model.predict(
    X_test[
        :,
        final_feature_indices,
    ]
)

final_elapsed_time = (
    time.time() - final_start
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions,
)

test_balanced_accuracy = (
    balanced_accuracy_score(
        y_test,
        test_predictions,
    )
)

test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0,
)

final_mlp = final_model.named_steps[
    "mlpclassifier"
]

final_summary_df = pd.DataFrame([
    {
        "Dataset": DATASET_NAME,
        "Feature Method": BEST_METHOD,
        "Selected Features": len(
            final_feature_indices
        ),
        "Features Removed": (
            FEATURES_REMOVED
        ),
        "Feature Reduction (%)": (
            FEATURE_REDUCTION_PERCENT
        ),
        "Model": "Baseline MLP 256-128",
        "Test Accuracy": test_accuracy,
        "Accuracy Change vs Baseline": (
            test_accuracy
            - BASELINE_ACCURACY
        ),
        "Test Balanced Accuracy": (
            test_balanced_accuracy
        ),
        "Balanced Accuracy Change vs Baseline": (
            test_balanced_accuracy
            - BASELINE_BALANCED_ACCURACY
        ),
        "Test Macro F1": test_macro_f1,
        "Macro F1 Change vs Baseline": (
            test_macro_f1
            - BASELINE_MACRO_F1
        ),
        "MLP Iterations": int(
            final_mlp.n_iter_
        ),
        "Final Train/Eval Time (s)": (
            final_elapsed_time
        ),
    }
])

display(final_summary_df)

,Dataset,Feature Method,Selected Features,Features Removed,Feature Reduction (%),Model,Test Accuracy,Accuracy Change vs Baseline,Test Balanced Accuracy,Balanced Accuracy Change vs Baseline,Test Macro F1,Macro F1 Change vs Baseline,MLP Iterations,Final Train/Eval Time (s)
0,CLAP 0.5s,All Features,1024,0,0.0,Baseline MLP 256-128,0.37621,0.00001,0.382821,0.000021,0.379008,0.000008,40,44.437313


In [21]:
classification_report_df = pd.DataFrame(
    classification_report(
        y_test,
        test_predictions,
        labels=TARGET_CLASSES,
        output_dict=True,
        zero_division=0,
    )
).transpose()

display(classification_report_df)

confusion_df = pd.DataFrame(
    confusion_matrix(
        y_test,
        test_predictions,
        labels=TARGET_CLASSES,
    ),
    index=[
        f"True: {label}"
        for label in TARGET_CLASSES
    ],
    columns=[
        f"Predicted: {label}"
        for label in TARGET_CLASSES
    ],
)

display(confusion_df)

,precision,recall,f1-score,support
correct,0.396414,0.387160,0.391732,1028.00000
arched_back,0.302667,0.338301,0.319493,671.00000
hunched_back,0.366404,0.358324,0.362319,907.00000
sideways,0.335427,0.311189,0.322854,858.00000
chest_breathing,0.284777,0.254695,0.268897,852.00000
over_articulation,0.486264,0.511561,0.498592,692.00000
under_articulation,0.462963,0.518519,0.489168,675.00000
accuracy,0.376210,0.376210,0.376210,0.37621
macro avg,0.376416,0.382821,0.379008,5683.00000
weighted avg,0.373456,0.376210,0.374279,5683.00000


,Predicted: correct,Predicted: arched_back,Predicted: hunched_back,Predicted: sideways,Predicted: chest_breathing,Predicted: over_articulation,Predicted: under_articulation
True: correct,398,115,155,107,109,58,86
True: arched_back,113,227,91,69,73,54,44
True: hunched_back,160,98,325,107,97,65,55
True: sideways,116,105,115,267,120,64,71
True: chest_breathing,97,110,99,136,217,89,104
True: over_articulation,56,52,50,56,78,354,46
True: under_articulation,64,43,52,54,68,44,350


In [22]:
selection_results_df.to_csv(
    OUTPUT_DIR
    / "all_selection_results.csv",
    index=False,
)

best_family_rows.to_csv(
    OUTPUT_DIR
    / "best_candidate_per_family.csv",
    index=False,
)

mlp_validation_results_df.to_csv(
    OUTPUT_DIR
    / "mlp_validation_results.csv",
    index=False,
)

final_summary_df.to_csv(
    OUTPUT_DIR
    / "final_test_result.csv",
    index=False,
)

classification_report_df.to_csv(
    OUTPUT_DIR
    / "classification_report.csv"
)

confusion_df.to_csv(
    OUTPUT_DIR
    / "confusion_matrix.csv"
)

prediction_df = pd.DataFrame({
    "Filename": test_groups,
    "True Label": y_test,
    "Predicted Label": (
        test_predictions
    ),
})

prediction_df.to_csv(
    OUTPUT_DIR
    / "test_predictions.csv",
    index=False,
)

np.savez_compressed(
    OUTPUT_DIR
    / "selected_feature_indices.npz",
    indices=final_feature_indices,
    method=np.asarray(
        BEST_METHOD
    ),
)

joblib.dump(
    {
        "dataset": DATASET_NAME,
        "feature_method": BEST_METHOD,
        "feature_indices": (
            final_feature_indices
        ),
        "model": final_model,
        "target_classes": (
            TARGET_CLASSES
        ),
    },
    OUTPUT_DIR
    / "final_model.joblib",
)

with open(
    OUTPUT_DIR
    / "best_configuration.json",
    "w",
) as file:
    json.dump(
        {
            "dataset": DATASET_NAME,
            "feature_method": (
                BEST_METHOD
            ),
            "selected_features": int(
                len(final_feature_indices)
            ),
            "features_removed": int(
                FEATURES_REMOVED
            ),
            "feature_reduction_percent": float(
                FEATURE_REDUCTION_PERCENT
            ),
            "test_accuracy": float(
                test_accuracy
            ),
            "test_balanced_accuracy": float(
                test_balanced_accuracy
            ),
            "test_macro_f1": float(
                test_macro_f1
            ),
        },
        file,
        indent=2,
    )

print(
    "Saved all results to:",
    OUTPUT_DIR.resolve(),
)

Saved all results to: /Users/bhavaykhatri/Desktop/msclap_2023/clap_0.5s_all_feature_selection
